In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as scp
import numpy as np

from scipy.stats import median_abs_deviation
from os import makedirs
from os.path import isfile

# Define pipeline functions 

In [ ]:
def dimensions_sparsity_check(expression_matrix_df: pd.DataFrame,
                              sample: str) -> None:
    no_genes, no_cells = expression_matrix_df.shape[0], expression_matrix_df.shape[1]
    
    print(f"For dataset {sample}")
    print(f"\t\u00b7# genes: {no_genes}")
    print(f"\t\u00b7# cells: {no_cells}")

    sparsity_coeff = round((expression_matrix_df == 0).sum().sum()/(no_genes*no_cells),2)*100

    print(f"\nSparsity coefficient: {sparsity_coeff}%")

def total_zeroes_filtering(expression_matrix_df: pd.DataFrame) -> pd.DataFrame:
    print(f"Number of genes with zero total expression: {(expression_matrix_df.sum(axis=1) == 0).sum()}")
    print(f"Number of cells consisting of non-expressed genes : {(expression_matrix_df.sum() == 0).sum()}")

    expression_matrix_df = expression_matrix_df.loc[expression_matrix_df.sum(axis=1) != 0] 
    expression_matrix_df = expression_matrix_df.loc[:,expression_matrix_df.sum(axis=0) != 0]

    return expression_matrix_df


def specific_type_RNA_filtering(expression_matrix_df: pd.DataFrame,
                                st_RNA_label: str,
                                st_ensgid: pd.Series,
                                st_RNA_threshold: float,
                                figure_output_file: str,
                                save_fig: bool = False) -> pd.DataFrame:
    if not st_RNA_label in ["mt_RNA","rRNA","apopt_RNA"]:
        raise ValueError("st_RNA_label can be assigned:'mt_RNA','rRNA' or 'apopt_RNA'")

    no_detected_st_genes = (expression_matrix_df.index.isin(st_ensgid)).sum()
    print(f"Number of detected {st_RNA_label} genes in data: {no_detected_st_genes}")

    if no_detected_st_genes == 0: return expression_matrix_df

    st_RNA_cell_percentage = expression_matrix_df.loc[expression_matrix_df.index.isin(st_ensgid)].sum()/expression_matrix_df.sum()

    st_RNA_cell_percentage.hist(bins=20,grid=False)

    plt.vlines(st_RNA_threshold,0,1500,colors="red",linestyles="dashed");

    plt.title(f"Percentile of {st_RNA_label} wrt cell total expression");
    plt.xlabel(f"{st_RNA_label} relative expression (%)");

    plt.yscale("log");

    if save_fig: plt.savefig(f"{figure_output_file}/{st_RNA_label}_filtering.png")
    
    print(f"Number of cells with excess of {st_RNA_label}: {(st_RNA_cell_percentage > st_RNA_threshold).sum()}")
    expression_matrix_df = expression_matrix_df.loc[:,st_RNA_cell_percentage <= st_RNA_threshold]

    return expression_matrix_df


def non_zero_cells(expression_matrix_df: pd.DataFrame,
                   non_zero_cells_threshold: int,
                   vline_lim: int,
                   figure_output_file: str,
                   save_fig: bool = False) -> pd.DataFrame:
    non_zero_cells_ser = expression_matrix_df.astype(bool).sum(axis=1)

    np.log10(non_zero_cells_ser).hist(bins=25,
                              grid=False);

    plt.vlines(np.log10(non_zero_cells_threshold),0,vline_lim,color="red",linestyles="dashed");

    plt.xlabel("$log_{10}(\\# non-zero\\ cells)$");

    if save_fig: plt.savefig(f"{figure_output_file}/non_zero_cells_filtering.png");

    print(f"Genes to be filtered out: {(non_zero_cells_ser <= non_zero_cells_threshold).sum()}");
    expression_matrix_df = expression_matrix_df.loc[non_zero_cells_ser > non_zero_cells_threshold]

    return expression_matrix_df


def logCPM_transform(expression_matrix_df: pd.DataFrame,
                     transformation: str) -> pd.DataFrame:
    if not transformation in ["log","CPM","logCPM"]:
        raise ValueError("trasformation can be assigned:'log','CPM','logCPM'")
    
    match transformation:
        case "log":
            return np.log10(1+expression_matrix_df)
        case "CPM":
            return expression_matrix_df/expression_matrix_df.sum()*1e6
        case "logCPM":
            return np.log10(1+(expression_matrix_df/expression_matrix_df.sum()*1e6))
        
        
def doublets_empty_droplets_MAD(expression_matrix_df: pd.DataFrame,
                                vline_log_lim: int,
                                vline_lim: int,
                                figure_output_file: str,
                                save_fig: bool = False) -> tuple[int,int]:
    count_depth_ser = expression_matrix_df.sum()
    
    count_depth_median = count_depth_ser.median()
    MAD = median_abs_deviation(count_depth_ser)

    lower_MAD = count_depth_median - 3*MAD
    upper_MAD = count_depth_median + 3*MAD

    print(f"Lower MAD threshold: {lower_MAD}")
    print(f"Upper MAD threshold: {upper_MAD}")

    print(f"Cells count depth below lower MAD threshold: {(count_depth_ser < lower_MAD).sum()}")
    print(f"Cells count depth above upper MAD threshold: {(count_depth_ser > upper_MAD).sum()}")

    fig, (ax1, ax2) = plt.subplots(1,2,figsize=(10,6))
    
    np.log10(count_depth_ser).hist(bins=30,grid=False,ax=ax1)

    ax1.vlines(np.log10(lower_MAD),0,vline_log_lim,color="red",linestyles="dashed")
    ax1.vlines(np.log10(upper_MAD),0,vline_log_lim,color="red",linestyles="dashed")

    ax1.set_xlabel("$log_{10}(count\\ depth)$")


    count_depth_ser.hist(bins=30,grid=False,ax=ax2)

    ax2.vlines(lower_MAD,0,vline_lim,color="red",linestyles="dashed")
    ax2.vlines(upper_MAD,0,vline_lim,color="red",linestyles="dashed")

    ax2.set_xlabel("count depth")
    
    if save_fig: fig.savefig(f"{figure_output_file}/MAD_log_non_corrected.png")

    return lower_MAD,upper_MAD


def call_scrublet(expression_matrix_df:pd.DataFrame) -> dict:
    adata_obj = scp.AnnData(expression_matrix_df.T)
    scrublet_doublets = {}

    for random_state in [0,42]:
        scp.pp.scrublet(adata_obj,
                        random_state=random_state)
        
        scrublet_doublets[random_state] = adata_obj.obs["predicted_doublet"][adata_obj.obs["predicted_doublet"] == True].index
    
    return scrublet_doublets

In [ ]:
parameters_dict = {"set_1":
                   {
                     "mt_RNA_threshold": 0.1,
                     "rRNA_threshold": 0.1,
                     "apoptosis_threshold": 0.1,
                     "non_zero_cells_threshold": 5
                   },
                   "set_2":
                   {
                     "mt_RNA_threshold": 0.05,
                     "rRNA_threshold": 0.05,
                     "apoptosis_threshold": 0.05,
                     "non_zero_cells_threshold": 3  
                   },
                   "set_3":{
                     "mt_RNA_threshold": 0.15,
                     "rRNA_threshold": 0.15,
                     "apoptosis_threshold": 0.15,
                     "non_zero_cells_threshold": 10 
                   }}

In [ ]:
output_dir = "" # Fill this with your local output dir

session_dir = f"{output_dir}/session_a"
makedirs(session_dir,exist_ok=True)

In [ ]:
sample = "pbmc.bead.enriched.sample.zheng"

data_file = f"./pbmc.csv.gz"
utility_file = f"./annotation.csv.gz"

In [ ]:
set_choice = "set_1"

In [ ]:
figure_output_dir = f"{session_dir}/figures"
HVGs_output_dir = f"{session_dir}/HVGs"
processed_data_output_dir = f"{session_dir}/processed_data"


for set_choice_key in parameters_dict.keys():
    makedirs(f"{figure_output_dir}/{set_choice_key}",exist_ok=True)
    makedirs(f"{HVGs_output_dir}/{set_choice_key}",exist_ok=True)
    makedirs(f"{processed_data_output_dir}/{set_choice_key}",exist_ok=True)

In [ ]:
expression_matrix_df = pd.read_csv(data_file,index_col=0)
ensembl_df = pd.read_csv(utility_file,index_col=0)

In [ ]:
expression_matrix_df.head(15)

In [ ]:
ensembl_df.head(15)

# Function-type pipeline

In [ ]:
dimensions_sparsity_check(expression_matrix_df,sample)

In [ ]:
expression_matrix_df = total_zeroes_filtering(expression_matrix_df)

In [ ]:
expression_matrix_df.sum().hist(bins=20,
                                grid=False);

plt.xlabel("Cell count depth");

if not isfile(f"{figure_output_dir}/cell_count_depth.png"): plt.savefig(f"{figure_output_dir}/cell_count_depth.png");

In [ ]:
expression_matrix_df.sum(axis=1).hist(bins=20,
                                grid=False);

plt.yscale("log");

plt.xlabel("# total transcripts per gene");

if not isfile(f"{figure_output_dir}/total_transcripts_per_cell.png"): plt.savefig(f"{figure_output_dir}/total_transcripts_per_cell.png")

In [ ]:
mitochondrial_ensgid = ensembl_df.loc[ensembl_df.MT_status != "NO_MT"].ensgid
apoptosis_ensgid = ensembl_df.loc[ensembl_df.APOPT_status != "NO_APOPT"].ensgid
ribosomal_ensgid = ensembl_df.loc[ensembl_df.RB_status != "NO_RB"].ensgid

In [ ]:
mt_RNA_threshold = parameters_dict[set_choice]["mt_RNA_threshold"]

expression_matrix_df = specific_type_RNA_filtering(expression_matrix_df,
                                                   "mt_RNA",
                                                   mitochondrial_ensgid,
                                                   mt_RNA_threshold,
                                                   f"{figure_output_dir}/{set_choice}",
                                                   save_fig=True)

In [ ]:
expression_matrix_df = total_zeroes_filtering(expression_matrix_df)

In [ ]:
rRNA_threshold = parameters_dict[set_choice]["rRNA_threshold"]

expression_matrix_df = specific_type_RNA_filtering(expression_matrix_df,
                                                   "rRNA",
                                                   ribosomal_ensgid,
                                                   rRNA_threshold,
                                                   f"{figure_output_dir}/{set_choice}")

In [ ]:
apoptosis_threshold = parameters_dict[set_choice]["apoptosis_threshold"]

expression_matrix_df = specific_type_RNA_filtering(expression_matrix_df,
                                                   "apopt_RNA",
                                                   apoptosis_ensgid,
                                                   apoptosis_threshold,
                                                   f"{figure_output_dir}/{set_choice}",
                                                   save_fig=True)

In [ ]:
expression_matrix_df = total_zeroes_filtering(expression_matrix_df)

In [ ]:
lower_MAD,upper_MAD = doublets_empty_droplets_MAD(expression_matrix_df,
                                                  500,
                                                  1200,
                                                  figure_output_dir)

In [ ]:
count_depth_ser = expression_matrix_df.sum()

high_cd_comp_index = count_depth_ser[count_depth_ser >= 3800].index
high_expression_matrix_df = expression_matrix_df.loc[:,high_cd_comp_index]

In [ ]:
_,upper_high_cd_MAD = doublets_empty_droplets_MAD(high_expression_matrix_df,
                                                  40,
                                                  60,
                                                  f"{figure_output_dir}/{set_choice}")

In [ ]:
np.log10(count_depth_ser).hist(bins=30,
                 grid=False);

plt.vlines(np.log10(lower_MAD),0,500,color="red",linestyles="dashed");
plt.vlines(np.log10(upper_high_cd_MAD),0,500,color="red",linestyles="dashed");

plt.xlabel("$log_{10}(count\\ depth)$");

if not isfile(f"{figure_output_dir}/{set_choice}/MAD_log_corrected.png"):
    plt.savefig(f"{figure_output_dir}/{set_choice}/MAD_log_corrected.png")

In [ ]:
scrublet_doublets_dict = call_scrublet(expression_matrix_df)

In [ ]:
scrublet_doublets_dict[0]

In [ ]:
count_depth_ser.loc[scrublet_doublets_dict[0]]

In [ ]:
scrublet_doublets_dict[42]

In [ ]:
count_depth_ser.loc[scrublet_doublets_dict[42]]

In [ ]:
expression_matrix_df = expression_matrix_df.loc[:,(count_depth_ser >= lower_MAD) & (count_depth_ser <= upper_high_cd_MAD)]

In [ ]:
expression_matrix_df = total_zeroes_filtering(expression_matrix_df)

In [ ]:
non_zero_cells_threshold = parameters_dict[set_choice]["non_zero_cells_threshold"]

aux_expression_matrix_df = non_zero_cells(expression_matrix_df,
                                      non_zero_cells_threshold,
                                      1800,
                                      f"{figure_output_dir}/{set_choice}",
                                      save_fig=True)

In [ ]:
expression_matrix_df = aux_expression_matrix_df

In [ ]:
expression_matrix_df = total_zeroes_filtering(expression_matrix_df)

In [ ]:
log_cpm_expression_matrix_df = logCPM_transform(expression_matrix_df,
                                             "logCPM")

In [ ]:
log_cpm_expression_matrix_df

In [ ]:
dimensions_sparsity_check(expression_matrix_df,sample)

In [ ]:
expression_matrix_df.to_csv(f"{processed_data_output_dir}/{set_choice}/filtered.raw.csv.gz",compression="gzip")
log_cpm_expression_matrix_df.to_csv(f"{processed_data_output_dir}/{set_choice}/filtered.logCPM.csv.gz",compression="gzip")

# HVGs

In [ ]:
plt.scatter(expression_matrix_df.mean(axis=1), y=expression_matrix_df.std(axis=1),s=2);

plt.xscale("log");
plt.yscale("log");

plt.xlabel("mean");
plt.ylabel("std");

In [ ]:
adata_obj = scp.AnnData(X=expression_matrix_df.T)

In [ ]:
adata_obj.layers["logCPM"] = log_cpm_expression_matrix_df.T

In [ ]:
scp.pp.highly_variable_genes(adata_obj,
                             layer="logCPM",
                             flavor="seurat")

In [ ]:
Seurat_mvp_HVGs_index = adata_obj.var['highly_variable'][adata_obj.var['highly_variable'] == True].index

In [ ]:
with open(f"{HVGs_output_dir}/{set_choice}/Seurat_mvp_HVGs.txt","w") as OUT:
    for nid,gene in enumerate(Seurat_mvp_HVGs_index):
        OUT.write(f"{nid}.{gene}\n")

In [ ]:
Seurat_mvp_HVGs_index

In [ ]:
scp.pp.highly_variable_genes(adata_obj,
                             flavor="seurat_v3")

In [ ]:
Seurat_vst_HVGs_index = adata_obj.var['highly_variable'][adata_obj.var['highly_variable'] == True].index

In [ ]:
Seurat_vst_HVGs_index

In [ ]:
with open(f"{HVGs_output_dir}/{set_choice}/Seurat_vst_HVGs.txt","w") as OUT:
    for nid,gene in enumerate(Seurat_vst_HVGs_index):
        OUT.write(f"{nid}.{gene}\n")